Overview & Teori Dasar
Pada bab ini, dipelajari tentang Convolutional Neural Networks (CNN) yang dirancang khusus untuk memproses data gambar secara efisien:

Reuse Weights (Pembagian Bobot): Menggunakan kernel/filter kecil yang digeser ke seluruh area gambar untuk mendeteksi pola lokal (seperti garis atau tepi) tanpa memerlukan terlalu banyak bobot.

Convolution Operation: Mengalikan matriks kernel dengan area lokal gambar (patch) secara bertahap.

Translational Invariance: Kemampuan model untuk mengenali fitur tertentu (seperti mata atau bentuk) terlepas dari lokasinya di dalam gambar.

1. Operasi Konvolusi Dasar (Convolution Layer)
Operasi konvolusi mengambil sub-area dari gambar (image patch) berukuran sama dengan kernel, melakukan perkalian titik (dot product), lalu menjumlahkannya menjadi satu nilai pada feature map.

In [1]:
import numpy as np

# Fungsi untuk mengambil potongan area lokal (patch) dari gambar
def get_image_section(layer, row, col, kernel_row, kernel_col):
    sub_section = layer[row:row+kernel_row, col:col+kernel_col]
    return sub_section

# Contoh Gambar Sederhana 8x8 (Pixel)
image = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0]
])

# Kernel/Filter ukuran 3x3 untuk pendeteksi tepi
kernel = np.array([
    [-1, -1, -1],
    [ 1,  1,  1],
    [ 0,  0,  0]
])

# Menghitung ukuran Feature Map hasil konvolusi
out_rows = image.shape[0] - kernel.shape[0] + 1
out_cols = image.shape[1] - kernel.shape[1] + 1
feature_map = np.zeros((out_rows, out_cols))

# Melakukan pergeseran kernel (sliding window)
for r in range(out_rows):
    for c in range(out_cols):
        patch = get_image_section(image, r, c, 3, 3)
        feature_map[r, c] = np.sum(patch * kernel)

print("Ukuran Feature Map:", feature_map.shape)
print("Hasil Feature Map:\n", feature_map)

Ukuran Feature Map: (6, 6)
Hasil Feature Map:
 [[ 2.  2.  1.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.]
 [-2. -1.  1.  2.  1.  0.]
 [ 0.  0.  0.  0.  0.  0.]
 [ 0. -1. -2. -2. -1.  0.]
 [ 0.  0.  0.  0.  0.  0.]]


Penjelasan Teoretis:

Kernel/Filter: Matriks kecil yang mendeteksi fitur spasial spesifik.

Feature Map: Peta representasi baru hasil ekstrak fitur lokal dari gambar masukan.

2. Arsitektur Sederhana Convolutional Neural Network (Forward Pass)

In [2]:
import numpy as np

def relu(x):
    return (x > 0) * x

np.random.seed(1)

# Input gambar 1 sampel (8x8)
input_image = image.reshape(1, 8, 8)

# Inisialisasi 16 Kernel acak berukuran 3x3
num_kernels = 16
kernel_rows = 3
kernel_cols = 3
kernels = 2 * np.random.random((num_kernels, kernel_rows, kernel_cols)) - 1

# Forward Pass Konvolusi + ReLU
sections = []
for r in range(out_rows):
    for c in range(out_cols):
        sections.append(get_image_section(input_image[0], r, c, kernel_rows, kernel_cols))

# Reshape sections agar dapat dikalikan secara matriks dengan seluruh kernel
sections = np.array(sections) # (out_rows*out_cols, 3, 3)
sections = sections.reshape(sections.shape[0], kernel_rows * kernel_cols)
kernels_flat = kernels.reshape(num_kernels, kernel_rows * kernel_cols)

# Perkalian matriks seluruh patch dengan seluruh kernel
conv_output = sections.dot(kernels_flat.T)
conv_output = conv_output.reshape(out_rows, out_cols, num_kernels)

# Aktivasi ReLU
conv_output = relu(conv_output)

print("Bentuk Output Layer Konvolusi:", conv_output.shape)

Bentuk Output Layer Konvolusi: (6, 6, 16)


Penjelasan Teoretis:

Shared Weights: Penggunaan kernel yang sama pada seluruh area gambar secara drastis mengurangi jumlah parameter bobot dibandingkan Fully Connected Layer biasa.

Multi-channel Feature Maps: Menggunakan banyak kernel sekaligus memungkinkan jaringan mempelajari berbagai macam fitur visual (seperti garis vertikal, horisontal, sudut, hingga tekstur) secara simultan.